# CICIoMT2024 — MI-RFE SVM, completed-stage resume across Colab sessions

Cell 13 starts or resumes ONE authorized MI-RFE SVM session. MI, full sklearn RF-RFE, completed k and final model are reused only after checksum/identity verification. SVC.fit/RFE/MI cannot resume in the middle. Do not use Run all. No Baseline SVM or RF training is launched.

New source/commit requires new unit evidence and new approval. Existing legacy .incomplete runs are not migrated. Labels: SCENARIO_PROXY; capture/session independence: NOT_ESTABLISHED.


## 1. Choose one MI-RFE SVM config, stable run name and new session ID

Set EXPECTED_COMMIT to the full SHA of the newly committed resumable code; the old c93eab commit does not contain this implementation. Keep RUN_NAME unchanged across sessions of the same experiment. Change SESSION_ID and approval for each session. All four FW/FS configurations are supported.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time

GITHUB_URL = 'https://github.com/kasum00/RFforNetwork.git'
EXPECTED_COMMIT = ''  # Paste the NEW full commit after pushing this resumable implementation
REPO = Path('/content/RFforNetwork_svm_resumable')
DRIVE_ROOT = Path('/content/drive/MyDrive/COMPACT-MI-RFE')
DRIVE_WINDOWS = Path('/content/drive/MyDrive/WINDOWS_CPLUS')
DRIVE_MANIFEST = Path('/content/drive/MyDrive/CSV_SPLIT_CPLUS')
SSD_ROOT = Path('/content/svm_data')
SSD_WINDOWS = SSD_ROOT / 'WINDOWS_CPLUS'
SSD_MANIFEST = SSD_ROOT / 'CSV_SPLIT_CPLUS'
LOCAL_RUNS = Path('/content/stage73b_runs')

# ===== CHỈNH BA DÒNG NÀY CHO MỖI RUN =====
EXPERIMENT = 'mi_rfe_svm'          # Dedicated MI-RFE SVM mode only
CONFIG = 'FW_512_FS_16'              # FW_256_FS_8 | FW_256_FS_16 | FW_512_FS_8 | FW_512_FS_16
RUN_ID = 'resumable01'
# ========================================
USE_SSD = True
CONFIG_SHORT = CONFIG.replace("FW_", "FW").replace("_FS_", "_FS")
RUN_NAME = f'{EXPERIMENT}_{CONFIG_SHORT}_{RUN_ID}'
assert EXPERIMENT == 'mi_rfe_svm'
assert CONFIG in ('FW_256_FS_8','FW_256_FS_16','FW_512_FS_8','FW_512_FS_16')
assert RUN_ID and all(ch.isascii() and (ch.isalnum() or ch in '-_') for ch in RUN_ID)
WINDOWS = SSD_WINDOWS if USE_SSD else DRIVE_WINDOWS
SESSION_ID = 'session01'  # New unique value for EVERY authorized session
RESUME = True
SESSION_BUDGET_SECONDS = 36000
HEARTBEAT_INTERVAL_SECONDS = 60
STOP_STARTING_MARGIN_SECONDS = 300
ENABLE_OFFICIAL_TRAINING = False
APPROVAL = DRIVE_ROOT/'approvals'/f'{RUN_NAME}.{SESSION_ID}.json'
UNIT_EVIDENCE = DRIVE_ROOT/'environment'/f'stage73b_unit_tests_{EXPECTED_COMMIT[:12]}.json'
GIB = 1024**3

def sha256_file(p: Path):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for buf in iter(lambda: f.read(8*1024*1024), b''):
            h.update(buf)
    return h.hexdigest()

print(json.dumps({'experiment':EXPERIMENT,'config':CONFIG,'run_name':RUN_NAME,
                  'windows_selected':str(WINDOWS),'official_train':'DISABLED'}, indent=2))

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
for p in [DRIVE_ROOT, DRIVE_WINDOWS, DRIVE_MANIFEST]:
    if not p.is_dir(): raise FileNotFoundError(f'Missing Drive folder: {p}')
for name in ('split_manifest.csv','validation.json'):
    if not (DRIVE_MANIFEST/name).is_file(): raise FileNotFoundError(DRIVE_MANIFEST/name)
print('Drive + manifest: PASS')

## 3. Clone and verify the NEW approved source

Only a newly created checkout is switched to EXPECTED_COMMIT. Existing checkouts must already be clean and match; no pull/reset/source patching.


In [ ]:
if len(EXPECTED_COMMIT) != 40 or any(ch not in '0123456789abcdef' for ch in EXPECTED_COMMIT.lower()):
    raise ValueError('Paste the NEW approved 40-character commit in Cell 1 first')
if not REPO.exists():
    subprocess.run(['git','clone',GITHUB_URL,str(REPO)],check=True)
    # Pin approved scientific source only in the checkout just created.
    subprocess.run(['git','-C',str(REPO),'switch','--detach',EXPECTED_COMMIT],check=True)
if not (REPO/'.git').is_dir():
    raise RuntimeError(f'{REPO} exists, but is not a Git checkout')
head = subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
dirty = subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip()
if dirty: raise RuntimeError('Repository is dirty; do not modify code under an existing approval')
if head != EXPECTED_COMMIT:
    raise RuntimeError(f'Git commit mismatch: {head}; expected {EXPECTED_COMMIT}. Update/push correct repo first; no auto-checkout.')
required = ('huyen_code/ciciomt24_training.py','huyen_code/ciciomt24_sliding_window.py',
            'colab/dataset_validator.py','colab/experiment_launcher.py','colab/svm_run_authorization.py',
            'colab/svm_official_supervisor.py','colab/setup_colab.py',
            'huyen_code/ciciomt24_svm_probe.py',
            'colab/svm_resumable_store.py','colab/svm_resumable_training.py',
            'colab/svm_resumable_authorization.py','colab/svm_resumable_execution.py')
missing = [p for p in required if not (REPO/p).is_file()]
if missing: raise FileNotFoundError('Missing Stage 7.3B code: '+', '.join(missing))
for p in [REPO, REPO/'huyen_code']:
    if str(p) not in sys.path: sys.path.insert(0,str(p))
print('Code checkout PASS:', head)

## 4. Kiểm tra thư viện khoa học (không âm thầm đổi môi trường)

Đã xác nhận trên Colab của em: NumPy `2.2.6`, pandas `2.3.3`, SciPy `1.15.3`, sklearn `1.7.2`, joblib `1.5.3`. Nếu thiếu, bật cờ cài đặt phía dưới và **restart session**, sau đó chạy lại từ cell 1. `pip check` có thể báo pandas khác dependency của `google-colab`; cần cân nhắc cảnh báo riêng.

In [ ]:
from colab.setup_colab import runtime_info
INSTALL_SCIENTIFIC_PROFILE = False  # Chỉ đổi thành True nếu profile hiện tại không PASS
if INSTALL_SCIENTIFIC_PROFILE:
    subprocess.run([sys.executable,'-m','pip','install','--only-binary=:all:',
                    'numpy==2.2.6','pandas==2.3.3','scipy==1.15.3',
                    'scikit-learn==1.7.2','joblib==1.5.3'],check=True)
    raise RuntimeError('Installation finished: RESTART COLAB RUNTIME, then run again from cell 1')
info=runtime_info()
print('Selected scientific profile matches:',info.get('scientific_versions_match_selected_profile'))
print('Installed packages:', info.get('packages') or info.get('versions'))
if not info.get('scientific_versions_match_selected_profile'):
    raise RuntimeError('Scientific environment mismatch. Review target versions before enabling INSTALL_SCIENTIFIC_PROFILE')

## 5. Kiểm tra tồn kho dữ liệu cửa sổ & dung lượng SSD

Không đọc dòng Test, không fit. Tất cả bốn cấu hình nằm trong cùng `WINDOWS_CPLUS`. Khi dữ liệu SSD đã có, chỉ kiểm tra dung lượng dự phòng 35 GiB; **không tính lại dung lượng của toàn bộ CSV đã sao chép**. Cell 6 vẫn bắt buộc kiểm tra SHA256 của dữ liệu cũ và dừng nếu thiếu/hỏng.

In [ ]:
import psutil
import ciciomt24_training as training
source_store=training.WindowStore(DRIVE_WINDOWS, split_root=DRIVE_MANIFEST)
source_files = sorted(set(f.path for files in source_store.files.values() for f in files))
csv_bytes = sum(p.stat().st_size for p in source_files)
other_bytes = sum(p.stat().st_size for p in DRIVE_WINDOWS.rglob('*') if p.is_file() and p not in set(source_files))
free = shutil.disk_usage('/content').free
print('CSV files:',len(source_files),'CSV GiB:',round(csv_bytes/GIB,2),
      'Other data GiB:',round(other_bytes/GIB,2),'Local disk free GiB:',round(free/GIB,2))
for config in training.CONFIGS:
    f=source_store.files[config]
    print(config,{role:sum(x.count for x in f if x.role==role) for role in ('train','val','test')})
print('SVC baseline params:',training.BASELINE_SVM_PARAMS)
print('SVC compact params:',training.COMPACT_SVM_PARAMS)
assert training.BASELINE_SVM_PARAMS['kernel']=='rbf' and training.BASELINE_SVM_PARAMS['C']==10.0
assert training.COMPACT_SVM_PARAMS['kernel']=='rbf' and training.COMPACT_SVM_PARAMS['C']==10.0
# Existing SSD staging consumes disk already. Only reserve space for a new copy.
# Cell 6 independently fails closed when existing staging/checksums are missing or corrupt.
copy_required = 0 if SSD_WINDOWS.exists() else csv_bytes + other_bytes
print('Additional SSD copy required GiB:', round(copy_required/GIB, 2))
if USE_SSD and free < copy_required + 35*GIB:
    raise RuntimeError('Not enough SSD headroom for verified staging plus training/cache. Do not start copying')
if USE_SSD and SSD_WINDOWS.exists():
    print('Existing staged WINDOWS detected; Cell 6 must verify every SHA256 before use')
print('SOURCE DATA INVENTORY: PASS')

## 6. Copy SSD và xác minh **toàn bộ SHA256** (fail-closed)

Cell 6 sửa lỗi `shutil.copytree` truyền chuỗi vào `verified_copy`; ghi manifest SHA256 của **từng file đã copy** tại SSD và Drive. **Mỗi lần tái sử dụng SSD, bắt buộc hash lại toàn bộ file SSD và đối chiếu với manifest.** Không có chế độ bỏ qua kiểm tra nội dung. Khi manifest thiếu, sai hoặc có `.copying`, cell dừng và yêu cầu kiểm tra/khôi phục; không tự ghi đè dữ liệu.

Dữ liệu nguồn Drive được hash trong lần copy đầu tiên. Để phát hiện nguồn Drive bị chỉnh nội dung sau đó (kể cả cùng kích thước), bật `VERIFY_DRIVE_SOURCE_CONTENT_ON_REUSE=True`, sẽ tốn thêm thời gian đọc lại toàn bộ Drive. Mặc định luôn kiểm tra SHA256 SSD, checksum summary + manifest nguồn và kích thước file nguồn.

In [ ]:
# CELL 6 — STAGE DATA + CHECKSUM MANIFEST, NOT MODEL TRAINING
STAGE_DATA_NOW = True
VERIFY_DRIVE_SOURCE_CONTENT_ON_REUSE = False  # True => hash toàn bộ CSV gốc Drive ở mỗi lần tái sử dụng
STAGING_SCHEMA = 'stage73b.ssd.file-sha256.v1'
LOCAL_SHA_MANIFEST = SSD_ROOT / 'ssd_file_sha256_manifest.json'
DRIVE_SHA_MANIFEST = DRIVE_ROOT / 'environment' / 'ciciomt24_windows_cplus_ssd_file_sha256_manifest.json'

# Inventories are relative paths, never absolute destinations; no Test rows are parsed.
source_items = []
for p in sorted(DRIVE_WINDOWS.rglob('*')):
    if p.is_file():
        source_items.append(('windows', p.relative_to(DRIVE_WINDOWS).as_posix(), p))
for name in ('split_manifest.csv', 'validation.json'):
    source_items.append(('manifest', name, DRIVE_MANIFEST / name))
source_lookup = {(area, rel): p for area, rel, p in source_items}
assert len(source_lookup) == len(source_items), 'Duplicate source relative paths'
assert set(source_files).issubset({p for area, rel, p in source_items if area == 'windows'})

def staging_path(area, rel):
    if not rel or Path(rel).is_absolute() or '..' in Path(rel).parts:
        raise ValueError(f'Unsafe relative path: {rel!r}')
    return (SSD_WINDOWS if area == 'windows' else SSD_MANIFEST) / rel

def build_staging_manifest(sha_by_file):
    rows = []
    for area, rel, p in source_items:
        key = (area, rel)
        if key not in sha_by_file:
            raise RuntimeError(f'Source SHA256 not captured during copy: {key}')
        rows.append({'area': area, 'path': rel, 'bytes': p.stat().st_size,
                     'sha256': sha_by_file[key]})
    return {'schema': STAGING_SCHEMA, 'files': rows,
            'source_window_summary_sha256': sha_by_file[('windows', 'window_run_summary.json')],
            'source_split_manifest_sha256': sha_by_file[('manifest', 'split_manifest.csv')],
            'source_validation_sha256': sha_by_file[('manifest', 'validation.json')]}

def verify_staging_contents(payload):
    if not isinstance(payload, dict) or payload.get('schema') != STAGING_SCHEMA:
        raise RuntimeError('Staging SHA256 manifest schema mismatch')
    records = payload.get('files')
    if not isinstance(records, list) or len(records) != len(source_items):
        raise RuntimeError('Staging manifest file count mismatch')
    indexed = {}
    for row in records:
        if not isinstance(row, dict) or not isinstance(row.get('sha256'), str):
            raise RuntimeError('Invalid staging manifest row')
        key = (row.get('area'), row.get('path'))
        if key in indexed:
            raise RuntimeError(f'Duplicate staging manifest key: {key}')
        indexed[key] = row
    if set(indexed) != set(source_lookup):
        raise RuntimeError('Staging manifest source inventory mismatch')
    actual_windows = {p.relative_to(SSD_WINDOWS).as_posix()
                      for p in SSD_WINDOWS.rglob('*') if p.is_file()}
    manifest_windows = {rel for area, rel in indexed if area == 'windows'}
    if actual_windows != manifest_windows:
        raise RuntimeError('Staged WINDOWS contains missing/extra files')
    for i, (area, rel, source) in enumerate(source_items, 1):
        row = indexed[(area, rel)]
        dst = staging_path(area, rel)
        size = row.get('bytes')
        if not isinstance(size, int) or size < 0 or not dst.is_file():
            raise RuntimeError(f'Missing/invalid SSD file: {dst}')
        if dst.stat().st_size != size or source.stat().st_size != size:
            raise RuntimeError(f'File size mismatch (Drive vs SSD vs manifest): {rel}')
        if sha256_file(dst) != row['sha256']:
            raise RuntimeError(f'SSD SHA256 MISMATCH: {dst}')
        if VERIFY_DRIVE_SOURCE_CONTENT_ON_REUSE and sha256_file(source) != row['sha256']:
            raise RuntimeError(f'Drive source SHA256 changed: {source}')
        if i % 40 == 0 or i == len(source_items):
            print(f'Checksum verified: {i}/{len(source_items)}', flush=True)
    for area, rel, recorded_key in (
        ('windows', 'window_run_summary.json', 'source_window_summary_sha256'),
        ('manifest', 'split_manifest.csv', 'source_split_manifest_sha256'),
        ('manifest', 'validation.json', 'source_validation_sha256')
    ):
        if indexed[(area, rel)]['sha256'] != payload.get(recorded_key):
            raise RuntimeError(f'Critical source metadata checksum differs: {rel}')
        if sha256_file(source_lookup[(area, rel)]) != payload[recorded_key]:
            raise RuntimeError(f'Critical Drive metadata changed: {rel}')
    return len(records)

# copytree(copy_function=...) supplies STR, not Path. Record the digest of source stream.
copy_hashes = {}
def verified_copy(source, target):
    source = Path(source)  # P1: required for shutil.copytree
    target = Path(target)  # P1: required for shutil.copytree
    target.parent.mkdir(parents=True, exist_ok=True)
    digest = hashlib.sha256()
    with source.open('rb') as reader, target.open('xb') as writer:
        for chunk in iter(lambda: reader.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
            writer.write(chunk)
    shutil.copystat(source, target)
    if target.stat().st_size != source.stat().st_size or sha256_file(target) != digest.hexdigest():
        raise OSError(f'Copy verification failed: {source} -> {target}')
    try:
        key = ('windows', source.relative_to(DRIVE_WINDOWS).as_posix())
    except ValueError:
        key = ('manifest', source.relative_to(DRIVE_MANIFEST).as_posix())
    if key in copy_hashes:
        raise RuntimeError(f'Duplicate copy record: {key}')
    copy_hashes[key] = digest.hexdigest()
    return str(target)

if USE_SSD:
    if not STAGE_DATA_NOW:
        raise RuntimeError('USE_SSD=True but STAGE_DATA_NOW=False: no verified SSD is available')
    SSD_ROOT.mkdir(parents=True, exist_ok=True)
    temp_win = SSD_ROOT / 'WINDOWS_CPLUS.copying'
    if temp_win.exists():
        raise FileExistsError(f'Interrupted staging requires manual inspection: {temp_win}')
    if not SSD_WINDOWS.exists():
        if LOCAL_SHA_MANIFEST.exists():
            raise RuntimeError('SHA256 manifest exists without SSD_WINDOWS; inspect state manually')
        if SSD_MANIFEST.exists() and any(SSD_MANIFEST.rglob('*')):
            raise RuntimeError('SSD manifest directory already present without staged windows; inspect state')
        start = time.monotonic()
        shutil.copytree(DRIVE_WINDOWS, temp_win, copy_function=verified_copy)
        for name in ('split_manifest.csv', 'validation.json'):
            verified_copy(DRIVE_MANIFEST / name, SSD_MANIFEST / name)
        SSD_WINDOWS.parent.mkdir(parents=True, exist_ok=True)
        temp_win.rename(SSD_WINDOWS)
        staged_payload = build_staging_manifest(copy_hashes)
        # Persist a source-anchored manifest. Existing Drive evidence is never replaced.
        DRIVE_SHA_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
        if DRIVE_SHA_MANIFEST.is_file():
            known = json.loads(DRIVE_SHA_MANIFEST.read_text(encoding='utf-8'))
            if known != staged_payload:
                raise RuntimeError('Existing Drive checksum manifest differs; do NOT overwrite')
        else:
            with DRIVE_SHA_MANIFEST.open('x', encoding='utf-8') as fh:
                json.dump(staged_payload, fh, indent=2, ensure_ascii=False)
        with LOCAL_SHA_MANIFEST.open('x', encoding='utf-8') as fh:
            json.dump(staged_payload, fh, indent=2, ensure_ascii=False)
        print('Copied and checksummed source in', round(time.monotonic()-start, 1), 'seconds')
    else:
        print('Existing SSD data detected: full SHA256 validation is mandatory')
    if not LOCAL_SHA_MANIFEST.is_file() or not DRIVE_SHA_MANIFEST.is_file():
        raise RuntimeError('SHA256 manifest missing. Staged data cannot be trusted/reused')
    local_checksum = json.loads(LOCAL_SHA_MANIFEST.read_text(encoding='utf-8'))
    drive_checksum = json.loads(DRIVE_SHA_MANIFEST.read_text(encoding='utf-8'))
    if local_checksum != drive_checksum:
        raise RuntimeError('Local SHA256 manifest differs from immutable Drive manifest')
    verified_count = verify_staging_contents(drive_checksum)
    staged_store = training.WindowStore(SSD_WINDOWS, split_root=SSD_MANIFEST)
    for config in training.CONFIGS:
        left, right = source_store.files[config], staged_store.files[config]
        if len(left) != len(right) or sum(f.count for f in left) != sum(f.count for f in right):
            raise RuntimeError(f'WindowStore inventory mismatch for {config}')
    print('SSD DATA READY — SHA256 VERIFIED:', verified_count, 'files')
    print('Local manifest:', LOCAL_SHA_MANIFEST)
    print('Drive manifest:', DRIVE_SHA_MANIFEST)
    if not VERIFY_DRIVE_SOURCE_CONTENT_ON_REUSE:
        print('NOTE: Drive CSV content is not rehashed on reuse; enable option if source may have changed')
else:
    print('Using Drive data directly. No SSD staging performed.')

## 7. Xác thực dataset & run identity trên đường dẫn đã chọn

Nếu dataset fingerprint/profile thay đổi vì đường dẫn mới, approval cũ không còn dùng được. Cell này fail-closed, không tự thay identity trong file approval.

In [ ]:
from colab.experiment_launcher import RunSpec
from colab.svm_run_authorization import collect_identity, expected_confirmation
from colab.svm_resumable_authorization import run_root
SPEC = RunSpec(EXPERIMENT, CONFIG, RUN_NAME)
identity, identity_evidence = collect_identity(REPO, WINDOWS, SPEC)
CONFIRMATION = expected_confirmation(identity)
RESUMABLE_ROOT = run_root(DRIVE_ROOT, SPEC)
print(json.dumps(identity, indent=2))
print('Run root:', RESUMABLE_ROOT)
print('Required confirmation:', CONFIRMATION)
print('Read-only identity collected; no authorization or training performed.')


## 8. User-run unit tests and evidence for this NEW source

The tests use mocks/small fixtures. Set RUN_RESUMABLE_UNIT_TESTS=True only when you want to execute them yourself. This cell creates evidence only after successful test execution and binds the exact commit/module hashes. Old Stage 7.3B evidence is insufficient for resumable execution.


In [ ]:
RUN_RESUMABLE_UNIT_TESTS = False
from colab.svm_resumable_authorization import extension_hashes
modules = [
    'colab.test_svm_resumable_store',
    'colab.test_svm_resumable_training',
    'colab.test_svm_resumable_execution',
    'colab.test_stage73b_authorization',
    'colab.test_stage73b_scientific_integrity',
]
if RUN_RESUMABLE_UNIT_TESTS:
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO), str(REPO/'huyen_code')])
    done = subprocess.run([sys.executable, '-m', 'unittest', '-v', *modules],
                          cwd=REPO, env=env, text=True, capture_output=True)
    print(done.stdout); print(done.stderr)
    if done.returncode != 0: raise RuntimeError('Tests failed; no PASS evidence written')
    unit = {
        'passed': True, 'returncode': 0, 'modules': modules,
        'git_commit': EXPECTED_COMMIT,
        'training_sha256': sha256_file(REPO/'huyen_code/ciciomt24_training.py'),
        'resumable_module_hashes': extension_hashes(REPO),
        'full_training_authorized': False,
    }
    UNIT_EVIDENCE.parent.mkdir(parents=True, exist_ok=True)
    if UNIT_EVIDENCE.exists():
        if json.loads(UNIT_EVIDENCE.read_text(encoding='utf-8')) != unit:
            raise FileExistsError('Different evidence exists; choose a new evidence path')
    else:
        with UNIT_EVIDENCE.open('x', encoding='utf-8') as handle:
            json.dump(unit, handle, indent=2)
if not UNIT_EVIDENCE.is_file(): raise FileNotFoundError(UNIT_EVIDENCE)
unit = json.loads(UNIT_EVIDENCE.read_text(encoding='utf-8'))
if not (unit.get('passed') is True and unit.get('returncode') == 0
        and unit.get('git_commit') == EXPECTED_COMMIT
        and unit.get('training_sha256') == sha256_file(REPO/'huyen_code/ciciomt24_training.py')
        and unit.get('resumable_module_hashes') == extension_hashes(REPO)
        and set(modules).issubset(unit.get('modules', []))):
    raise RuntimeError('Missing/stale resumable unit-test evidence')
print('Resumable source unit evidence: PASS; training is still disabled.')


## 8A. **Tùy chọn:** tạo Train-only technical probe khi cấu hình chưa có bằng chứng

Nếu đã có probe hợp lệ cho cấu hình hiện chọn, **bỏ qua cell này**. Với cấu hình mới, đổi `RUN_TRAIN_ONLY_PROBE=True` rồi chỉ chạy cell này một lần. Probe chỉ fit mẫu Train tối đa 8.192 hàng, **không** chạy Validation/Test, không phải training chính thức. Output được lưu Drive, không ghi đè. Chạy xong tiếp tục cell 9 để xác minh.

In [ ]:
RUN_TRAIN_ONLY_PROBE = False
PROBE_SIZE = 8192
if not RUN_TRAIN_ONLY_PROBE:
    print('Train-only probe not requested in this session')
else:
    from colab.dataset_validator import manifest_root_for
    manifest_root=manifest_root_for(WINDOWS,REPO)
    probe_script=REPO/'huyen_code/ciciomt24_svm_probe.py'
    if not probe_script.is_file(): raise FileNotFoundError(probe_script)
    probe_name=f'{RUN_NAME}_probe8192_technical.json'
    local_probe=Path('/content')/probe_name
    drive_probe=DRIVE_ROOT/'technical_probes'/probe_name
    if local_probe.exists() or drive_probe.exists():
        raise FileExistsError('Technical probe exists: do not overwrite; inspect existing evidence')
    cmd=[sys.executable,str(probe_script),'probe',
         '--repo',str(REPO),'--windows',str(WINDOWS),
         '--manifest-root',str(manifest_root),'--experiment',EXPERIMENT,
         '--config',CONFIG,'--output',str(local_probe),
         '--sizes',str(PROBE_SIZE),'--seed','42',
         '--timeout-per-size','300','--max-rss-gib','6']
    env=os.environ.copy()
    env['PYTHONPATH']=os.pathsep.join([str(REPO),str(REPO/'huyen_code'),env.get('PYTHONPATH','')])
    rc=subprocess.run(cmd,cwd=REPO,env=env,check=False).returncode
    if not local_probe.is_file(): raise RuntimeError(f'Probe output missing; exit={rc}')
    result=json.loads(local_probe.read_text(encoding='utf-8'))
    if rc!=0 or result.get('status')!='PASS_TECHNICAL_ONLY':
        raise RuntimeError('Probe incomplete/failed — no successful evidence saved to Drive')
    DRIVE_PROBE_DIR=drive_probe.parent
    DRIVE_PROBE_DIR.mkdir(parents=True,exist_ok=True)
    with local_probe.open('rb') as src, drive_probe.open('xb') as dst:
        shutil.copyfileobj(src,dst,length=8*1024*1024)
    if sha256_file(drive_probe)!=sha256_file(local_probe):
        raise RuntimeError('Technical probe Drive copy checksum mismatch')
    print('TECHNICAL-ONLY PROBE PASS:',drive_probe)
    print('SHA256:',sha256_file(drive_probe))
    print('OFFICIAL TRAINING: NOT STARTED')

## 9. Chọn technical evidence hợp lệ, **có fallback**

Nếu có file 64K ưu tiên và file đó còn hợp lệ thì chọn nó. Nếu thiếu/hỏng/không đúng identity, tự tìm trong `technical_probes` (bao gồm file 8.192 mẫu vừa được tạo ở bước 8A); mọi ứng viên đều phải qua `_validate_technical_evidence(EXACT_CONFIG_PROBE)`. Nếu không có file nào hợp lệ thì **dừng**, không tự cho phép huấn luyện.

In [ ]:
from colab.svm_run_authorization import _validate_technical_evidence
PROBE_DIR = DRIVE_ROOT / 'technical_probes'
# Để None => tìm tự động; muốn cố định một file thì điền tên file chính xác.
TECHNICAL_EVIDENCE_OVERRIDE = None
PREFERRED_EVIDENCE_NAME = (
    'baseline_svm_FW512_FS16_scaling64k_01.json'
    if (EXPERIMENT, CONFIG) == ('baseline_svm', 'FW_512_FS_16') else None
)

def observed_source_identity(data):
    source = data.get('source_hashes') if isinstance(data.get('source_hashes'), dict) else {}
    return {'experiment': data.get('experiment'), 'config': data.get('config'),
            'training_code_sha256': data.get('training_code_sha256') or source.get('training'),
            'window_summary_sha256': data.get('window_summary_sha256') or source.get('summary')}

def validate_probe_candidate(p):
    data = json.loads(p.read_text(encoding='utf-8'))
    observed = observed_source_identity(data)
    policy = {'technical_evidence': {'policy': 'EXACT_CONFIG_PROBE',
              'evidence_sha256': sha256_file(p), 'evidence_identity': observed}}
    result = _validate_technical_evidence(policy, p, identity)
    return data, policy['technical_evidence'], result

if not PROBE_DIR.is_dir():
    raise FileNotFoundError(f'No technical_probes directory: {PROBE_DIR}')
if TECHNICAL_EVIDENCE_OVERRIDE:
    # Explicit choice must be exact, no silent substitution.
    requested = Path(TECHNICAL_EVIDENCE_OVERRIDE)
    if requested.name != str(requested):
        raise ValueError('Specify only the evidence filename, not an external path')
    candidates = [PROBE_DIR / requested.name]
else:
    candidates = []
    if PREFERRED_EVIDENCE_NAME:
        candidates.append(PROBE_DIR / PREFERRED_EVIDENCE_NAME)
    # Deterministic fallback: include new probe from 8A and any other files in the directory.
    candidates.extend(sorted(PROBE_DIR.glob('*.json')))
    candidates = list(dict.fromkeys(candidates))

valid, errors = [], []
for p in candidates:
    if not p.is_file():
        errors.append((p.name, 'missing'))
        continue
    try:
        data, policy, result = validate_probe_candidate(p)
        valid.append((p, data, policy, result))
    except (OSError, ValueError, KeyError, PermissionError, TypeError) as exc:
        errors.append((p.name, f'{type(exc).__name__}: {exc}'))

if not valid:
    print('No valid exact-identity technical evidence. Errors:', errors[:12])
    raise RuntimeError('Technical evidence missing/invalid; do not create approval')

preferred = next((row for row in valid
                  if row[0].name == PREFERRED_EVIDENCE_NAME), None) if not TECHNICAL_EVIDENCE_OVERRIDE else None
if preferred:
    chosen = preferred
else:
    # Largest successful bounded sample, filename tie-break; never accept another config.
    chosen = max(valid, key=lambda row: (max(row[1].get('sizes') or [0]), row[0].name))
TECHNICAL_EVIDENCE, probe_data, technical_policy, technical_result = chosen
print('CHOSEN EVIDENCE:', TECHNICAL_EVIDENCE)
print('SHA256:', technical_result['sha256'])
print('Sizes:', probe_data.get('sizes'))
print('Valid candidates:', len(valid), '| Invalid/missing:', len(errors))
if errors:
    print('Rejected examples:', errors[:5])
print('EXACT_CONFIG_PROBE: PASS; OFFICIAL TRAINING: NOT STARTED')

## 10. Inactive per-session approval draft

36,000 seconds is a proposed session budget, not an estimate of SVC completion. No new stage/k starts within the last 300 seconds. The supervisor may terminate an unfinished fit at the hard deadline; that fit has no resumable optimizer state.


In [ ]:
from colab.svm_resumable_authorization import blocked_template
budgets = {
    'max_elapsed_seconds': SESSION_BUDGET_SECONDS,
    'max_tree_rss_bytes': 28*GIB,
    'min_start_available_ram_bytes': 42*GIB,
    'min_runtime_available_ram_bytes': 10*GIB,
    'min_free_disk_bytes': 30*GIB,
    'poll_interval_seconds': 10,
}
draft = blocked_template(REPO, WINDOWS, SPEC)
draft['budgets'] = budgets
draft['technical_evidence'] = technical_policy
draft['resume_policy'].update(
    session_id=SESSION_ID, session_budget_seconds=SESSION_BUDGET_SECONDS,
    stop_starting_new_work_margin_seconds=STOP_STARTING_MARGIN_SECONDS,
    heartbeat_interval_seconds=HEARTBEAT_INTERVAL_SECONDS,
)
# If the previous runtime died with a lock, confirm termination manually first.
# Do not set this true while that runtime may still be alive.
draft['interrupted_session_takeover'] = {
    'previous_session_id': '',
    'previous_session_terminated_confirmed': False,
}
DRAFT = DRIVE_ROOT/'approval_drafts'/f'{RUN_NAME}.{SESSION_ID}.BLOCKED.json'
DRAFT.parent.mkdir(parents=True, exist_ok=True)
if DRAFT.exists():
    if json.loads(DRAFT.read_text(encoding='utf-8')) != draft:
        raise FileExistsError('Different draft exists; never overwrite')
else:
    with DRAFT.open('x', encoding='utf-8') as handle: json.dump(draft, handle, indent=2)
print(json.dumps(draft, indent=2))
print('BLOCKED template only:', DRAFT)


## 11. Explicit human approval for this session

Review exact identity, all resource limits, completed-stage restart policy and single-Test constraint. An old approval cannot open this mode. If a previous Colab runtime died with a lock, only confirm its termination after you have verified it is no longer running. This cell creates a new approval only after explicit inputs.


In [ ]:
HUMAN_REVIEW_AND_CREATE_APPROVAL = False
if not HUMAN_REVIEW_AND_CREATE_APPROVAL:
    print('BLOCKED: no effective approval created.')
else:
    print(json.dumps(draft, indent=2))
    print('Run:', RUN_NAME, 'Session:', SESSION_ID)
    if input('Type ACCEPT COMPLETED-STAGE RESUME AND SINGLE FINAL TEST: ').strip() != 'ACCEPT COMPLETED-STAGE RESUME AND SINGLE FINAL TEST':
        raise PermissionError('Explicit scientific/restart acknowledgement missing')
    if input('Type the exact AUTHORIZE phrase from Cell 7: ').strip() != CONFIRMATION:
        raise PermissionError('Run/source authorization phrase mismatch')
    session_phrase = f'RESUME {RUN_NAME} {SESSION_ID}'
    if input(f'Type {session_phrase}: ').strip() != session_phrase:
        raise PermissionError('Session-specific acknowledgement missing')
    final_approval = json.loads(json.dumps(draft))
    final_approval.update(
        decision='APPROVED_STAGE73B', effective=True, confirmation=CONFIRMATION,
        scientific_contract_acknowledged=True,
        test_is_single_final_evaluation_acknowledged=True,
        capture_session_independence_not_established_acknowledged=True,
    )
    final_approval['resume_policy'].update(
        allow_completed_stage_resume=True, repeat_only_uncommitted_work=True,
        never_repeat_claimed_test=True,
    )
    APPROVAL.parent.mkdir(parents=True, exist_ok=True)
    if APPROVAL.exists():
        if json.loads(APPROVAL.read_text(encoding='utf-8')) != final_approval:
            raise FileExistsError('Different approval exists; never overwrite')
    else:
        with APPROVAL.open('x', encoding='utf-8') as handle:
            json.dump(final_approval, handle, indent=2)
    print('Explicit approval stored:', APPROVAL, 'No training started.')


## 12. Read-only Stage 7.3B authorization with explicit resume policy

Checks current source/dataset/runtime, full-data admission and new test evidence. A resume requires the same scientific identity; SESSION_ID/approval/budget may change only through a new explicit approval. No model is fitted and no Test feature row is read.


In [ ]:
from colab.svm_resumable_authorization import authorize_resume
AUTH_CONTEXT = None
if not APPROVAL.is_file():
    print('BLOCKED: missing effective session approval:', APPROVAL)
else:
    AUTH_CONTEXT = authorize_resume(
        REPO, WINDOWS, LOCAL_RUNS, DRIVE_ROOT, SPEC, APPROVAL, TECHNICAL_EVIDENCE,
        CONFIRMATION, unit_test_evidence=unit, session_id=SESSION_ID,
        resume=RESUME, session_budget_seconds=SESSION_BUDGET_SECONDS,
        heartbeat_interval_seconds=HEARTBEAT_INTERVAL_SECONDS,
    )
    print(json.dumps(AUTH_CONTEXT.base.public_record(), indent=2))
    print(json.dumps(AUTH_CONTEXT.policy, indent=2))
    print('AUTHORIZATION PASS. No training or Test feature read performed.')


## 13. START / RESUME — one authorized MI-RFE SVM session

ENABLE_OFFICIAL_TRAINING is False by default in Cell 1. Enable it only after Cell 12 succeeds and you decide to start this session. Live stage/k/CPU/RSS/elapsed output appears every 60 seconds. A successful safe pause is not a completed experiment: create a new session approval, keep the run name, and resume at the first unfinished k. Never restart Test if a durable Test claim exists without a completed final-Test checkpoint.


In [ ]:
# CELL 13 — Controlled completed-stage resume; no automatic retries.
if not ENABLE_OFFICIAL_TRAINING:
    print('MI-RFE SVM DISABLED — no worker started.')
else:
    if EXPERIMENT != 'mi_rfe_svm' or AUTH_CONTEXT is None:
        raise PermissionError('Only the explicitly authorized MI-RFE SVM context is supported')
    command = [
        sys.executable, '-m', 'colab.svm_resumable_execution', 'supervise',
        '--repo', str(REPO), '--windows', str(WINDOWS),
        '--local-runs', str(LOCAL_RUNS), '--drive-root', str(DRIVE_ROOT),
        '--config', CONFIG, '--run-name', RUN_NAME,
        '--approval', str(APPROVAL), '--technical-evidence', str(TECHNICAL_EVIDENCE),
        '--unit-test-evidence', str(UNIT_EVIDENCE), '--confirmation', CONFIRMATION,
        '--session-id', SESSION_ID, '--session-budget-seconds', str(SESSION_BUDGET_SECONDS),
        '--heartbeat-interval-seconds', str(HEARTBEAT_INTERVAL_SECONDS),
    ]
    if RESUME: command.append('--resume')
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO), str(REPO/'huyen_code')])
    print('Starting ONE authorized MI-RFE SVM session:', RUN_NAME, SESSION_ID, flush=True)
    done = subprocess.run(command, cwd=REPO, env=env, check=False)
    if done.returncode:
        raise RuntimeError('Session stopped/failed. Inspect durable logs and Test claim before any new session.')
    print('Session ended. Read Cell 14: a safe pause is not a final result.')


## 14. Read-only resume status / final verification

Only inspect progress, session metadata and checkpoint checksums. No estimator is fitted or loaded for prediction, and Test is never evaluated again here.


In [ ]:
from colab.svm_resumable_store import CheckpointStore, read_json, file_sha
print('Run:', RUN_NAME, 'Config:', CONFIG, 'Session:', SESSION_ID)
if not RESUMABLE_ROOT.is_dir():
    print('NOT STARTED')
else:
    for path in (
        RESUMABLE_ROOT/'progress.json',
        RESUMABLE_ROOT/'sessions'/SESSION_ID/'session_result.json',
    ):
        if path.is_file(): print(path, json.dumps(read_json(path), indent=2))
    manifest = read_json(RESUMABLE_ROOT/'run_manifest.json')
    checkpoints = CheckpointStore(RESUMABLE_ROOT, manifest['binding'], resume=True)
    if manifest['binding']['identity'] != identity:
        raise RuntimeError('Current source/data/run identity differs from checkpoint')
    if checkpoints.load('final_test') is not None:
        final_ref = checkpoints.reference('final')
        checkpoints.load_required('final_test', parents={'final':final_ref})
        completed = RESUMABLE_ROOT/'results'/'completed'
        if not (completed/'artifact_manifest.json').is_file():
            print('FINAL TEST COMMITTED; publication needs verified recovery. DO NOT EVALUATE TEST AGAIN.')
        else:
            # This branch already has a completed directory: verify/reuse only.
            from colab.svm_resumable_training import publish_results
            completed = publish_results(checkpoints)
            result = read_json(completed/'result.json')
            if result.get('test_evaluation_calls') != 1: raise RuntimeError('Invalid Test count')
            print('FINAL RESULT:', json.dumps(result, indent=2))
            print('Artifact integrity verified; no Test evaluation performed by this cell.')
    else:
        claim = RESUMABLE_ROOT/'results'/'test_evaluation_claim.json'
        print('INCOMPLETE / PAUSED. Test claim exists:', claim.exists())
        if claim.exists(): print('MANUAL REVIEW REQUIRED: do not repeat Test.')


## 15. Resume limits

Completed MI, full RFE, individual k and final model bundles are reusable after SHA256/parent validation. Local memmaps are reused only when their identity and checksums match; a new runtime reloads Train/Validation CSVs but reuses MI/RFE/k. No large arrays are copied to Drive.

An unfinished SVC.fit, MI or RFE restarts from the beginning of that uncommitted operation under a new session approval. A final SVC fit must finish in one session. A Test claim without final_test completion blocks automatic recovery. Never use Test to select features, k or resource policy.
